# Setup

In [26]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()

assert (REPO_ROOT / "src" / "stock_investment_dss").exists(), (
    f"Notebook must be run from repo root. Current cwd: {REPO_ROOT}"
)

# Required for `python -m stock_investment_dss...`
os.environ["PYTHONPATH"] = "src"


def load_env_file(path, *, override=False, allowed_prefixes=None):
    path = Path(path)

    if not path.exists():
        print(f"Skipped missing env file: {path}")
        return

    loaded = 0

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")

        if allowed_prefixes is not None:
            if not any(key.startswith(prefix) for prefix in allowed_prefixes):
                continue

        if override or key not in os.environ:
            os.environ[key] = value
            loaded += 1

    print(f"Loaded {loaded} values from {path}")


# Load general project env safely.
# IMPORTANT:
# This is not the source of truth for IQN hyperparameters.
# The clean experiment manifest controls IQN config.
load_env_file(".env", override=False)

# Load W&B-related settings.
# We allow both:
# - standard WANDB_* keys
# - project-specific STOCK_INVESTMENT_DSS_WANDB_* keys
#
# IMPORTANT:
# We do NOT force WANDB_MODE=online here.
# The existing project/run logic should decide how W&B is initialized.
load_env_file(
    ".env.wandb.local",
    override=True,
    allowed_prefixes=("WANDB_", "STOCK_INVESTMENT_DSS_WANDB_"),
)

print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

for key in [
    "WANDB_MODE",
    "WANDB_PROJECT",
    "WANDB_ENTITY",
    "WANDB_API_KEY",
    "STOCK_INVESTMENT_DSS_WANDB_ENABLED",
    "STOCK_INVESTMENT_DSS_WANDB_PROJECT",
    "STOCK_INVESTMENT_DSS_WANDB_ENTITY",
]:
    value = os.environ.get(key)
    if key == "WANDB_API_KEY" and value:
        value = "***"
    print(f"{key} =", value)

Loaded 0 values from .env
Loaded 4 values from .env.wandb.local
Repo root: c:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN
Python: c:\Users\gurug\miniconda3\envs\stockdss\python.exe
PYTHONPATH: src
WANDB_MODE = online
WANDB_PROJECT = StockInvestmentDSS
WANDB_ENTITY = guldmand-SDU
WANDB_API_KEY = ***
STOCK_INVESTMENT_DSS_WANDB_ENABLED = true
STOCK_INVESTMENT_DSS_WANDB_PROJECT = StockInvestmentDSS
STOCK_INVESTMENT_DSS_WANDB_ENTITY = guldmand-SDU


# Ensure W&B works

In [8]:
from pathlib import Path
from datetime import datetime
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim

try:
    import wandb
except ImportError as exc:
    raise RuntimeError("wandb is not installed in this conda env.") from exc


# ---------------------------------------------------------------------
# W&B smoke-test run folder
# ---------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
run_dir = Path("outputs") / "runs" / f"{timestamp}_wandb_relu_smoke_test"
run_dir.mkdir(parents=True, exist_ok=True)

project = os.environ.get("STOCK_INVESTMENT_DSS_WANDB_PROJECT", "StockInvestmentDSS")
entity = os.environ.get("STOCK_INVESTMENT_DSS_WANDB_ENTITY") or None
enabled = os.environ.get("STOCK_INVESTMENT_DSS_WANDB_ENABLED", "").lower() == "true"

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("W&B enabled:", enabled)
print("W&B project:", project)
print("W&B entity:", entity)
print("Run dir:", run_dir.resolve())
print("WANDB_API_KEY:", "***" if os.environ.get("WANDB_API_KEY") else None)

if not enabled:
    raise RuntimeError("STOCK_INVESTMENT_DSS_WANDB_ENABLED is not true.")

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("WANDB_API_KEY is missing.")


# ---------------------------------------------------------------------
# Tiny ReLU network smoke test
# ---------------------------------------------------------------------

torch.manual_seed(42)

x = torch.randn(256, 10)
true_w = torch.randn(10, 1)
y = x @ true_w + 0.1 * torch.randn(256, 1)

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


wandb_run = wandb.init(
    project=project,
    entity=entity,
    name=f"wandb_relu_smoke_test_{timestamp}",
    group="wandb-smoke-tests",
    job_type="relu_smoke_test",
    config={
        "test_type": "5_epoch_relu_nn",
        "epochs": 5,
        "optimizer": "Adam",
        "lr": 1e-3,
        "loss": "MSELoss",
        "input_dim": 10,
        "hidden_dim": 32,
        "output_dim": 1,
        "run_dir": str(run_dir.resolve()),
    },
    dir=str(run_dir),
    tags=["wandb-smoke-test", "relu", "not-thesis-result"],
)

for epoch in range(1, 6):
    optimizer.zero_grad()
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()

    loss_value = float(loss.item())
    print(f"Epoch {epoch}/5 - loss={loss_value:.6f}")

    wandb_run.log(
        {
            "epoch": epoch,
            "train/loss": loss_value,
        },
        step=epoch,
    )

wandb_run.finish()

print("\nDone.")
print("Expected local W&B folder under:")
print(run_dir.resolve())
print("\nCheck root wandb folder should ideally be False:")
print("Root ./wandb exists:", Path("wandb").exists())

Python: c:\Users\gurug\miniconda3\envs\stockdss\python.exe
Torch: 2.10.0
W&B enabled: True
W&B project: StockInvestmentDSS
W&B entity: guldmand-SDU
Run dir: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN\outputs\runs\2026_05_23_081425_wandb_relu_smoke_test
WANDB_API_KEY: ***


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: guldmand (guldmand-SDU) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/5 - loss=7.324364
Epoch 2/5 - loss=7.284550
Epoch 3/5 - loss=7.245087
Epoch 4/5 - loss=7.205954
Epoch 5/5 - loss=7.167115


epoch,▁▃▅▆█
train/loss,█▆▄▃▁
epoch,5
train/loss,7.16712



Done.
Expected local W&B folder under:
C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN\outputs\runs\2026_05_23_081425_wandb_relu_smoke_test

Check root wandb folder should ideally be False:
Root ./wandb exists: False


In [9]:
import wandb
print(wandb.run)

None


# IQN Hold Diagnostics

In [11]:
# Run clean 25k baseline with updated manifest and logger
# first verify the experiment config
!python -u -m compileall src/stock_investment_dss

Listing 'src/stock_investment_dss'...
Listing 'src/stock_investment_dss\\baselines'...
Listing 'src/stock_investment_dss\\data'...
Listing 'src/stock_investment_dss\\decision'...
Listing 'src/stock_investment_dss\\dss'...
Listing 'src/stock_investment_dss\\environments'...
Listing 'src/stock_investment_dss\\evaluation'...
Listing 'src/stock_investment_dss\\evaluation\\configs'...
Listing 'src/stock_investment_dss\\experiment_tracking'...
Listing 'src/stock_investment_dss\\rl'...
Listing 'src/stock_investment_dss\\rl\\agents'...
Listing 'src/stock_investment_dss\\rl\\config'...
Listing 'src/stock_investment_dss\\rl\\models'...
Listing 'src/stock_investment_dss\\rl\\nets'...
Listing 'src/stock_investment_dss\\rl\\replay_buffers'...
Listing 'src/stock_investment_dss\\runner'...
Listing 'src/stock_investment_dss\\strategies'...
Listing 'src/stock_investment_dss\\strategies\\predefined'...
Listing 'src/stock_investment_dss\\uncertainty'...
Listing 'src/stock_investment_dss\\utilities'...


In [12]:
# verify the experiment
!python -u -m stock_investment_dss.runner.verify_experiment_config --config configs/experiments/clean_25k_baseline_v1.json

[verify_experiment_config] Loading manifest: configs\experiments\clean_25k_baseline_v1.json
[verify] No pre-existing IQN env vars to clear.
[verify] Applied 46 env_overrides from manifest.

[verify] --- IQN Config Assertion ---
  [PASS] batch_size: expected=64  actual=64
  [PASS] replay_capacity: expected=100000  actual=100000
  [PASS] target_update_interval: expected=1000  actual=1000
  [PASS] num_tau_samples: expected=32  actual=32
  [PASS] num_tau_prime_samples: expected=32  actual=32
  [PASS] num_action_quantiles: expected=32  actual=32
  [PASS] epsilon_decay_steps: expected=15000  actual=15000
  [PASS] epsilon_start: expected=1.0  actual=1.0
  [PASS] epsilon_final: expected=0.05  actual=0.05
  [PASS] epsilon_eval: expected=0.0  actual=0.0
  [PASS] use_layer_norm: expected=True  actual=True
  [PASS] hidden_dim: expected=128  actual=128
  [PASS] cosine_embedding_dim: expected=64  actual=64
  [PASS] gamma: expected=0.99  actual=0.99
  [PASS] kappa: expected=1.0  actual=1.0
  [PASS] t

In [13]:
# run the experiment
!python -u -m stock_investment_dss.runner.run_iqn_experiment_from_config --config configs/experiments/clean_25k_baseline_v1.json

[run_iqn_experiment_from_config] Experiment: clean_25k_baseline_v1
[run_iqn_experiment_from_config] Manifest:   configs\experiments\clean_25k_baseline_v1.json
[run_iqn_experiment_from_config] Applied 46 env overrides.
[run_iqn_experiment_from_config] Running config verification...
[verify_experiment_config] Loading manifest: configs\experiments\clean_25k_baseline_v1.json
[verify] Cleared 23 pre-existing IQN env vars: ['STOCK_INVESTMENT_DSS_IQN_BATCH_SIZE', 'STOCK_INVESTMENT_DSS_IQN_REPLAY_CAPACITY', 'STOCK_INVESTMENT_DSS_IQN_TARGET_UPDATE_INTERVAL', 'STOCK_INVESTMENT_DSS_IQN_NUM_TAU_SAMPLES', 'STOCK_INVESTMENT_DSS_IQN_NUM_TAU_PRIME_SAMPLES', 'STOCK_INVESTMENT_DSS_IQN_NUM_ACTION_QUANTILES', 'STOCK_INVESTMENT_DSS_IQN_EPSILON_START', 'STOCK_INVESTMENT_DSS_IQN_EPSILON_FINAL', 'STOCK_INVESTMENT_DSS_IQN_EPSILON_DECAY_STEPS', 'STOCK_INVESTMENT_DSS_IQN_EPSILON_EVAL', 'STOCK_INVESTMENT_DSS_IQN_HIDDEN_DIM', 'STOCK_INVESTMENT_DSS_IQN_COSINE_EMBEDDING_DIM', 'STOCK_INVESTMENT_DSS_IQN_LEARNING_RATE'

2026-05-23 08:28:51,561 | INFO | stock_investment_dss.system | Starting StockInvestmentDSS IQN learning curve multiseed launcher.
2026-05-23 08:28:51,561 | INFO | stock_investment_dss.system | Project root: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN
2026-05-23 08:28:51,567 | INFO | stock_investment_dss.run.2026_05_23_082851_d_iqn_dss_iqn_learning_curve_multiseed_launcher | Created run directory: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN\outputs\runs\2026_05_23_082851_d_iqn_dss_iqn_learning_curve_multiseed_launcher
2026-05-23 08:28:51,567 | INFO | stock_investment_dss.run.2026_05_23_082851_d_iqn_dss_iqn_learning_curve_multiseed_launcher | Run id: 2026_05_23_082851_d_iqn_dss_iqn_learning_curve_multiseed_launcher
2026-05-23 08:28:51,567 | INFO | stock_investment_dss.run.2026_05_23_082851_d_iqn_dss_iqn_learning_curve_multiseed_launcher | Seeds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
2026-05-23 08:28:51,567 | INFO | stock_investment_dss.run.2026_05_23_082

# V2 run_finrl_smoke_test_demo_10_new
Run from terminal  
python scripts/finrl_smoke_test_demo_10_new_v2.py

or via python cell below  

In [24]:
!python -u scripts/finrl_smoke_test_demo_10_new_v2.py

[OK] Frozen data file found: data\market\daily\imports\market_data_demo10_new_2010_2026.csv

FinRL Baseline Suite Smoke Test - demo_10_new

Repo root:    C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN
Python:       c:\Users\gurug\miniconda3\envs\stockdss\python.exe
Start time:   2026-05-24 21:52:54

Environment variables applied:
  PYTHONPATH = src
  STOCK_INVESTMENT_DSS_DAILY_DATA_UNIVERSE = demo_10_new
  STOCK_INVESTMENT_DSS_DAILY_DATASET_ID = demo_10_new_long_2010_2026_finrl_smoke
  STOCK_INVESTMENT_DSS_DAILY_DATA_START = 2010-01-01
  STOCK_INVESTMENT_DSS_DAILY_DATA_END = 2026-12-31
  STOCK_INVESTMENT_DSS_DAILY_DATA_IMPORT_FILE = data/market/daily/imports/market_data_demo10_new_2010_2026.csv
  STOCK_INVESTMENT_DSS_DAILY_DATA_USE_CACHE = true
  STOCK_INVESTMENT_DSS_DAILY_DATA_ALLOW_DOWNLOAD = false
  STOCK_INVESTMENT_DSS_FORCE_FINRL_DOWNLOAD = false
  STOCK_INVESTMENT_DSS_FINRL_TICKERS = COST,AVGO,LLY,ORCL,CAT,BA,KO,MCD,WMT,PG
  STOCK_INVESTMENT_DSS_PIT_SPLIT_ID = de

2026-05-24 21:52:57,093 | INFO | stock_investment_dss.system | Starting StockInvestmentDSS FinRL baseline suite smoke test.
2026-05-24 21:52:57,093 | INFO | stock_investment_dss.system | Project root: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN
2026-05-24 21:52:57,109 | INFO | stock_investment_dss.run.2026_05_24_215257_d_iqn_dss_finrl_baseline_suite_smoke_test | Created run directory: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN\outputs\runs\2026_05_24_215257_d_iqn_dss_finrl_baseline_suite_smoke_test
2026-05-24 21:52:57,110 | INFO | stock_investment_dss.run.2026_05_24_215257_d_iqn_dss_finrl_baseline_suite_smoke_test | Run id: 2026_05_24_215257_d_iqn_dss_finrl_baseline_suite_smoke_test
2026-05-24 21:52:57,110 | INFO | stock_investment_dss.run.2026_05_24_215257_d_iqn_dss_finrl_baseline_suite_smoke_test | Dataset id: demo_10_new_long_2010_2026_finrl_smoke
2026-05-24 21:52:57,110 | INFO | stock_investment_dss.run.2026_05_24_215257_d_iqn_dss_finrl_base

# V2 run_finrl_smoke_test_demo_10_new
Run from terminal  

```
python scripts/finrl_multiseed_demo_10_new.py
```

or via python cell below  

In [25]:
!python -u scripts/finrl_multiseed_demo_10_new.py

[OK] Frozen data file found: data\market\daily\imports\market_data_demo10_new_2010_2026.csv

FinRL Baseline MULTISEED Run - demo_10_new

Repo root:    C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN
Python:       c:\Users\gurug\miniconda3\envs\stockdss\python.exe
Start time:   2026-05-24 21:56:41

Configuration:
  Seeds:           1,2,3,4,5
  Agents:          a2c,ddpg,td3,ppo,sac
  Include MVO:     true
  Timesteps:       25000
  Tickers:         COST,AVGO,LLY,ORCL,CAT,BA,KO,MCD,WMT,PG
  PIT date:        2024-01-01
  Trade end:       2026-12-31
  Run summary:     true

Estimated runtime: 60-90 minutes (5 seeds × 25k steps × 6 agents)
Cancel-safe: completed seeds are preserved on Ctrl+C
----------------------------------------------------------------------


Multiseed run finished - Duration: 58.8 min (3529 sec)
Return code: 0

Latest multiseed launcher run: 2026_05_24_215642_d_iqn_dss_finrl_baseline_multiseed_launcher
--- Launcher summary (first 30 lines) ---
{
  "statu

2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Starting StockInvestmentDSS FinRL baseline multiseed launcher.
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Project root: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Created run directory: C:\Users\gurug\Dropbox\DataScience\Speciale\D-IQN-DSS\FinRL_IQN\outputs\runs\2026_05_24_215642_d_iqn_dss_finrl_baseline_multiseed_launcher
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Run id: 2026_05_24_215642_d_iqn_dss_finrl_baseline_multiseed_launcher
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Seeds: [1, 2, 3, 4, 5]
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Agents: a2c,ddpg,td3,ppo,sac
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Include MVO: True
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Stop on failure: False
2026-05-24 21:56:42 | INFO | stock_investment_dss.run | Run summary after: True
2026-0